# 08_v5c_3_1_long_history — V5C 3.1 长周期完整测试

> 目标：用**最长可得数据**测试 V5C 3.1，验证在 2008 GFC + 2000 dot-com 等极端环境下的表现

## V5C 3.1 配置

```
进攻 50%:  VOO 15% / QQQ 15% / HQH 10% / XLV 10%
对冲 30%:  GLDM 20% / BCX 10%
防御 20%:  VGSH 20%
```

## 长周期代理映射

| 标的 | 代理 | 起始 |
|---|---|---|
| VOO | VFINX | 1976 |
| QQQ | QQQ | 1999-03 |
| HQH | HQH | 1987 |
| XLV | XLV | 1998-12 |
| GLDM | GLD (2004+) → GC=F 黄金期货 (1999+) | 1999 |
| BCX | DBC (2006+) → PCRIX (2002+) | 2002 |
| VGSH | VFITX | 1991 |

## 测试窗口

1. **窗口 B: 2006+ (~20 年)**：完整覆盖 2008 GFC，所有代理直接数据可用
2. **窗口 A: 2000+ (~26 年)**：扩展尝试覆盖 2000-2002 dot-com，使用 GC=F + PCRIX 代理

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 拉取所有可能用到的代理
# ============================================================
all_tickers = [
    'VFINX',   # VOO 代理 (1976+)
    'QQQ',     # 1999+
    'HQH',     # 1987+
    'XLV',     # 1998+
    'VFITX',   # VGSH 代理 (1991+)
    'GLD',     # GLDM 代理 (2004+)
    'GC=F',    # 黄金期货 (1999+, 用于 2004 之前 GLD 替代)
    'DBC',     # BCX 代理 (2006+)
    'PCRIX',   # 商品 (2002+, 用于 2006 之前 DBC 替代)
]

raw = yf.download(all_tickers, start='1999-01-01', auto_adjust=True)['Close']

# 检查每个 ticker 数据起始日
print('每个 ticker 数据起始日期:')
for t in all_tickers:
    if t in raw.columns:
        first_valid = raw[t].first_valid_index()
        last_valid = raw[t].last_valid_index()
        n_days = raw[t].notna().sum()
        print(f'  {t:<10}: {first_valid.date() if first_valid else "None":<14} -> {last_valid.date() if last_valid else "None"}  ({n_days} 天)')
    else:
        print(f'  {t:<10}: 未拉取到数据')

In [ ]:
# ============================================================
# 构造长史合成系列
# ============================================================
# GLDM 长史: GLD (2004+) 优先, GC=F 反向延伸到 1999
# 关键: 在 GLD 上市当天用 GLD 价格 / GC=F 价格作为换算系数, 把更早的 GC=F 拉伸到 GLD 价格水平

def synthesize_long_history(short_series, long_series, name='合成'):
    """
    short_series: 短史精确数据 (例如 GLD)
    long_series: 长史代理 (例如 GC=F)
    返回: 合成序列, 短史区间用 short, 之前用比例换算的 long
    """
    short = short_series.dropna().copy()
    long = long_series.dropna().copy()
    if len(short) == 0:
        return long
    overlap_start = short.index[0]
    if overlap_start <= long.index[0]:
        return short  # 短史本身就够长
    
    # 找 short 起始日附近 long 的值
    long_at_overlap = long.loc[:overlap_start].iloc[-1] if not long.loc[:overlap_start].empty else long.iloc[0]
    short_at_overlap = short.iloc[0]
    scale = short_at_overlap / long_at_overlap
    
    # 把 long 中早于 overlap_start 的部分按 scale 缩放
    early = long.loc[:overlap_start].iloc[:-1] * scale
    combined = pd.concat([early, short])
    combined = combined.sort_index()
    combined = combined[~combined.index.duplicated(keep='last')]
    print(f'  {name}: long 区段 {early.index.min().date()}->{early.index.max().date()} ({len(early)} 天) + short {short.index.min().date()}->{short.index.max().date()} ({len(short)} 天)')
    return combined

print('构造长史合成数据:')
gold_long = synthesize_long_history(raw['GLD'], raw['GC=F'], 'Gold')
commod_long = synthesize_long_history(raw['DBC'], raw['PCRIX'], 'Commodity')

# 构造完整数据
long_data = pd.DataFrame({
    'VOO': raw['VFINX'],
    'QQQ': raw['QQQ'],
    'HQH': raw['HQH'],
    'XLV': raw['XLV'],
    'VGSH': raw['VFITX'],
    'GLDM': gold_long,
    'BCX': commod_long,
})

# 查看每个标的最早起始日
print('\n构造后各标的起始:')
for c in long_data.columns:
    fv = long_data[c].first_valid_index()
    print(f'  {c}: {fv.date() if fv else "None"}')

# 找到所有标的都有数据的最早日期
common_start = long_data.dropna().index[0]
print(f'\n所有标的共同起始日: {common_start.date()}')
long_data_full = long_data.dropna()
print(f'共同期间: {long_data_full.index[0].date()} -> {long_data_full.index[-1].date()}  ({len(long_data_full)} 天, {len(long_data_full)/252:.1f} 年)')

In [ ]:
# ============================================================
# simulate + metrics 函数
# ============================================================
def simulate_rebalance(returns_df, target_weights, threshold_pp=5.0):
    used = [t for t in target_weights.keys() if t in returns_df.columns]
    sub_returns = returns_df[used]
    target = np.array([target_weights[t] for t in used])
    target = target / target.sum()
    current_weights = target.copy()
    portfolio_returns = []
    rebalance_dates = [sub_returns.index[0]]
    
    for date, daily_ret in sub_returns.iterrows():
        port_ret = np.sum(current_weights * daily_ret.values)
        portfolio_returns.append(port_ret)
        new_weights = current_weights * (1 + daily_ret.values)
        new_weights = new_weights / new_weights.sum()
        max_dev_pp = np.max(np.abs(new_weights - target)) * 100
        if max_dev_pp >= threshold_pp:
            current_weights = target.copy()
            rebalance_dates.append(date)
        else:
            current_weights = new_weights
    
    return pd.Series(portfolio_returns, index=sub_returns.index), rebalance_dates

def compute_metrics(returns_series, rebalance_dates, name='Portfolio'):
    cum = (1 + returns_series).cumprod()
    n_years = len(returns_series) / 252
    cagr = cum.iloc[-1] ** (1/n_years) - 1
    vol = returns_series.std() * np.sqrt(252)
    sharpe = (cagr - 0.04) / vol
    downside = returns_series[returns_series < 0]
    sortino = (cagr - 0.04) / (downside.std() * np.sqrt(252))
    rolling_max = cum.expanding().max()
    drawdown = (cum / rolling_max) - 1
    max_dd = drawdown.min()
    max_dd_date = drawdown.idxmin()
    calmar = cagr / abs(max_dd)
    return {
        'Name': name, 'CAGR': cagr, 'Vol': vol,
        'Sharpe': sharpe, 'Sortino': sortino,
        'Max DD': max_dd, 'Max DD Date': max_dd_date,
        'Calmar': calmar, 'Rebalances': len(rebalance_dates) - 1,
        'Years': n_years,
    }

V5C_3_1 = {
    'VOO': 0.15, 'QQQ': 0.15, 'HQH': 0.10, 'XLV': 0.10,
    'GLDM': 0.20, 'BCX': 0.10, 'VGSH': 0.20,
}
print(f'V5C 3.1 权重和: {sum(V5C_3_1.values()):.2f}')

In [ ]:
# ============================================================
# 长史完整回测
# ============================================================
returns_long = long_data_full.pct_change().dropna()

ret_long, dates_long = simulate_rebalance(returns_long, V5C_3_1, threshold_pp=5.0)
metrics_long = compute_metrics(ret_long, dates_long, f'V5C 3.1 ({returns_long.index[0].year}-{returns_long.index[-1].year})')

print('=' * 78)
print(f'V5C 3.1 长史完整回测 ({metrics_long["Years"]:.1f} 年)')
print('=' * 78)
for col in ['CAGR', 'Vol', 'Sharpe', 'Sortino', 'Max DD', 'Calmar']:
    fmt = '{:.2%}' if col not in ['Sharpe','Sortino','Calmar'] else '{:.3f}'
    print(f'  {col:<10}: {fmt.format(metrics_long[col])}')
print(f'  Max DD 日期: {metrics_long["Max DD Date"].date()}')
print(f'  Rebalances: {metrics_long["Rebalances"]} 次 (平均 {metrics_long["Years"]/metrics_long["Rebalances"]:.1f} 年/次)')

# 与短期 (14.6Y) 对比
returns_short = returns_long.loc['2011-01-01':]
if len(returns_short) > 0:
    ret_short, dates_short = simulate_rebalance(returns_short, V5C_3_1, threshold_pp=5.0)
    metrics_short = compute_metrics(ret_short, dates_short, 'V5C 3.1 (2011+)')
    print('\n--- 与之前 14.6 年回测对比 ---')
    for col in ['CAGR', 'Vol', 'Sharpe', 'Max DD', 'Calmar']:
        fmt = '{:.2%}' if col not in ['Sharpe','Calmar'] else '{:.3f}'
        print(f'  {col:<10}: 长史 {fmt.format(metrics_long[col])} vs 短期 {fmt.format(metrics_short[col])}')

In [ ]:
# ============================================================
# 净值与回撤可视化 (长史)
# ============================================================
cum_long = (1 + ret_long).cumprod()
rm_long = cum_long.expanding().max()
dd_long = (cum_long / rm_long) - 1

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

axes[0].plot(cum_long, linewidth=2, color='C0')
axes[0].set_title(f'V5C 3.1 长史净值曲线 ({metrics_long["Years"]:.1f} 年, log scale, 起始=1.0)', fontsize=14)
axes[0].set_yscale('log')
axes[0].grid(alpha=0.3)
axes[0].set_ylabel('Cumulative Return')

# 标注危机
crises_marks = {
    'Dot-com': ('2000-03-01', '2002-10-01'),
    '2008 GFC': ('2007-10-01', '2009-03-01'),
    'COVID': ('2020-02-19', '2020-04-30'),
    '2022 Bear': ('2022-01-01', '2022-10-31'),
}
for name, (s, e) in crises_marks.items():
    s_d = pd.Timestamp(s)
    e_d = pd.Timestamp(e)
    if s_d > cum_long.index[0]:
        axes[0].axvspan(s_d, e_d, alpha=0.15, color='red')
        axes[0].text(s_d, cum_long.max() * 0.85, name, rotation=45, fontsize=9)

axes[1].fill_between(dd_long.index, dd_long.values, 0, alpha=0.5, color='C3')
axes[1].set_title('回撤曲线', fontsize=14)
axes[1].grid(alpha=0.3)
axes[1].set_ylabel('Drawdown')
axes[1].set_xlabel('Date')
for name, (s, e) in crises_marks.items():
    s_d = pd.Timestamp(s)
    e_d = pd.Timestamp(e)
    if s_d > dd_long.index[0]:
        axes[1].axvspan(s_d, e_d, alpha=0.15, color='red')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 危机期分析
# ============================================================
crises = {
    'Dot-com 崩盘 (2000-2002)':  ('2000-03-01', '2002-10-09'),
    '2008 GFC':                  ('2007-10-09', '2009-03-09'),
    '2008 GFC 详细':              ('2008-09-01', '2009-03-09'),
    '2018-Q4 跌势':                ('2018-10-01', '2018-12-24'),
    '2020 COVID':                 ('2020-02-19', '2020-04-30'),
    '2022 Bear':                  ('2022-01-01', '2022-10-12'),
    '2025 Q1 (Tariff)':           ('2025-01-01', '2025-04-30'),
}

print('=' * 78)
print('V5C 3.1 在历史危机中的表现')
print('=' * 78)
print(f'{"危机":<26} {"时长":<8} {"区间回报":<12} {"区间内 Max DD":<14}')
print('-' * 78)
for name, (s, e) in crises.items():
    if pd.Timestamp(s) < ret_long.index[0]:
        print(f'{name:<26} 数据不可用')
        continue
    sub = ret_long.loc[s:e]
    if len(sub) == 0:
        continue
    ret = (1 + sub).prod() - 1
    days = len(sub)
    cum_sub = (1 + sub).cumprod()
    rm = cum_sub.expanding().max()
    dd = (cum_sub / rm) - 1
    max_dd = dd.min()
    print(f'{name:<26} {days:>4} 天  {ret:>+10.2%}    {max_dd:>+10.2%}')

In [ ]:
# ============================================================
# 与基准对比: V5C 3.1 vs VOO 单一指数 vs 60/40
# ============================================================
# VOO (VFINX) 单一
voo_ret = returns_long['VOO']
metrics_voo = compute_metrics(voo_ret, [voo_ret.index[0]], 'VOO 单一持有')

# 60/40 (60% VOO + 40% VGSH)
weights_6040 = {'VOO': 0.60, 'VGSH': 0.40}
ret_6040, dates_6040 = simulate_rebalance(returns_long, weights_6040, threshold_pp=5.0)
metrics_6040 = compute_metrics(ret_6040, dates_6040, '60/40 经典')

# 全股票分散 (VOO+QQQ+XLV+HQH 各 25%)
weights_eqstock = {'VOO': 0.25, 'QQQ': 0.25, 'XLV': 0.25, 'HQH': 0.25}
ret_eqstock, dates_eqstock = simulate_rebalance(returns_long, weights_eqstock, threshold_pp=5.0)
metrics_eqstock = compute_metrics(ret_eqstock, dates_eqstock, '全股票分散')

df = pd.DataFrame([metrics_long, metrics_voo, metrics_6040, metrics_eqstock])
df = df[['Name', 'CAGR', 'Vol', 'Sharpe', 'Sortino', 'Max DD', 'Calmar']].set_index('Name')
df['CAGR'] = df['CAGR'].apply(lambda x: f'{x:.2%}')
df['Vol'] = df['Vol'].apply(lambda x: f'{x:.2%}')
df['Sharpe'] = df['Sharpe'].apply(lambda x: f'{x:.3f}')
df['Sortino'] = df['Sortino'].apply(lambda x: f'{x:.3f}')
df['Max DD'] = df['Max DD'].apply(lambda x: f'{x:.2%}')
df['Calmar'] = df['Calmar'].apply(lambda x: f'{x:.3f}')

print('=' * 78)
print('V5C 3.1 vs 基准对比 (长史完整周期)')
print('=' * 78)
print(df)

# 净值曲线对比
fig, ax = plt.subplots(figsize=(14, 6))
for ret_s, label in [(ret_long, 'V5C 3.1'), (voo_ret, 'VOO'), (ret_6040, '60/40'), (ret_eqstock, '全股票分散')]:
    cum = (1 + ret_s).cumprod()
    ax.plot(cum, label=label, linewidth=2, alpha=0.85)
ax.set_title(f'V5C 3.1 vs 基准 ({metrics_long["Years"]:.1f} 年, log scale)')
ax.set_yscale('log')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 滚动 5 年 Sharpe (回测期内表现稳定性)
# ============================================================
window = 252 * 5
rolling_cagr = (1 + ret_long).rolling(window).apply(lambda x: x.prod() ** (252/len(x)) - 1, raw=False)
rolling_vol = ret_long.rolling(window).std() * np.sqrt(252)
rolling_sharpe = (rolling_cagr - 0.04) / rolling_vol

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(rolling_cagr * 100, color='C0', linewidth=2, label='滚动 5Y CAGR')
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_title('5 年滚动 CAGR (%)')
axes[0].set_ylabel('CAGR (%)')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(rolling_sharpe, color='C2', linewidth=2, label='滚动 5Y Sharpe')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].axhline(0.5, color='gray', linewidth=0.5, linestyle='--')
axes[1].axhline(1.0, color='gray', linewidth=0.5, linestyle='--')
axes[1].set_title('5 年滚动 Sharpe')
axes[1].set_ylabel('Sharpe')
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

# 滚动 Sharpe 统计
valid_sharpe = rolling_sharpe.dropna()
print(f'\n滚动 5Y Sharpe 统计:')
print(f'  最小: {valid_sharpe.min():.3f}  ({valid_sharpe.idxmin().date()})')
print(f'  最大: {valid_sharpe.max():.3f}  ({valid_sharpe.idxmax().date()})')
print(f'  中位: {valid_sharpe.median():.3f}')
print(f'  Sharpe < 0 的时间占比: {(valid_sharpe < 0).mean():.1%}')
print(f'  Sharpe > 0.5 的时间占比: {(valid_sharpe > 0.5).mean():.1%}')
print(f'  Sharpe > 1.0 的时间占比: {(valid_sharpe > 1.0).mean():.1%}')

## 解读模板

运行后填空：

**长史完整指标**（覆盖 dot-com + 2008 + COVID + 2022）：
1. CAGR: ____
2. Sharpe: ____
3. Max DD: ____ (日期: ____)
4. Calmar: ____

**关键危机表现**：
- Dot-com (2000-2002): ____
- 2008 GFC: ____
- 2020 COVID: ____
- 2022 Bear: ____

## V5C 3.1 是否通过最终验证？

**通过标准**：
- ✅ 长史 Max DD < -30%（即组合在 2008 GFC 这种最坏情境下仍可控）
- ✅ 长史 Sharpe ≥ 0.5（风险调整后回报合理）
- ✅ 在所有危机中**没有任何一次**让组合下跌超过 -35%
- ✅ 滚动 5Y Sharpe 中位数 > 0.5
- ✅ 长史 CAGR ≥ 8%（合理的长期复合增长）

**否决条件**：
- ❌ Max DD 接近 VOO 单一持有（说明对冲层失效）
- ❌ Sharpe < 0.5（无法证明 hedging 的价值）
- ❌ 任何 5 年滚动窗口内 Sharpe 长期为负